# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

import os
from typing import List, Optional
from pydantic import BaseModel, Field

from openai import OpenAI

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
DOC_URL = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
DOC_TITLE_HINT = "Managing Oneself"

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(DOC_URL)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

len(document_text), len(docs)

(51452, 13)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="<= 1 paragraph: why relevant for an AI professional")
    Summary: str = Field(description="Concise summary <= 1000 tokens")
    Tone: str = Field(description="The specific style used in the summary")
    InputTokens: int
    OutputTokens: int

In [4]:
TONE = "Bureaucratese" 

DEVELOPER_INSTRUCTIONS = """
You are a careful summarization assistant.
Return ONLY valid structured output that matches the provided schema.
Do not include extra keys.

Hard constraints:
- Summary must be no longer than 1000 tokens.
- Relevance must be no longer than 1 paragraph.
- The summary MUST clearly use the specified tone.
- Stay faithful to the source; do not invent facts.
"""

USER_PROMPT_TEMPLATE = """
Summarize the following article.

Tone to use: {tone}

Context (the article text):
{article_text}
""".strip()

In [5]:
client = OpenAI(
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

user_prompt = USER_PROMPT_TEMPLATE.format(tone=TONE, article_text=document_text)

response = client.responses.parse(
    model="gpt-4o-mini", 
    input=[
        {"role": "developer", "content": DEVELOPER_INSTRUCTIONS},
        {"role": "user", "content": user_prompt},
    ],
    text_format=ArticleSummary,
)

summary_obj: ArticleSummary = response.output_parsed

summary_obj, response.usage

(ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='This article provides foundational insights into self-management for knowledge workers, which is crucial for anyone in artificial intelligence and technology roles, where understanding personal strengths and how to leverage them is essential for career success and impact.', Summary="In 'Managing Oneself,' Peter Drucker asserts that the success of knowledge workers is contingent upon their self-awareness regarding strengths, weaknesses, values, and optimal work environments. He argues that individuals must take responsibility for their careers, effectively becoming their own CEOs. This entails placing emphasis on self-reflection through feedback analysis—where one records expected outcomes of key decisions and later compares them to actual results. Identifying personal strengths guides individuals towards roles where they can excel, while understanding their values ensures compatibility with organizational c

In [6]:

usage = getattr(response, "usage", None)

input_tokens = None
output_tokens = None

if usage:
    input_tokens = getattr(usage, "input_tokens", None)
    output_tokens = getattr(usage, "output_tokens", None)

    if input_tokens is None and hasattr(usage, "prompt_tokens"):
        input_tokens = usage.prompt_tokens
    if output_tokens is None and hasattr(usage, "completion_tokens"):
        output_tokens = usage.completion_tokens

summary_obj.InputTokens = int(input_tokens) if input_tokens is not None else -1
summary_obj.OutputTokens = int(output_tokens) if output_tokens is not None else -1

summary_obj

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='This article provides foundational insights into self-management for knowledge workers, which is crucial for anyone in artificial intelligence and technology roles, where understanding personal strengths and how to leverage them is essential for career success and impact.', Summary="In 'Managing Oneself,' Peter Drucker asserts that the success of knowledge workers is contingent upon their self-awareness regarding strengths, weaknesses, values, and optimal work environments. He argues that individuals must take responsibility for their careers, effectively becoming their own CEOs. This entails placing emphasis on self-reflection through feedback analysis—where one records expected outcomes of key decisions and later compares them to actual results. Identifying personal strengths guides individuals towards roles where they can excel, while understanding their values ensures compatibility with organizational cu

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [7]:
from deepeval.models import GPTModel

judge_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

In [8]:
class EvalResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str

    CoherenceScore: float
    CoherenceReason: str

    TonalityScore: float
    TonalityReason: str

    SafetyScore: float
    SafetyReason: str

In [9]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams

# ----- Summarization metric with bespoke questions (>= 5) -----
summarization_questions = [
    "Does the summary capture the article’s central thesis about managing oneself (strengths, values, contribution)?",
    "Does the summary accurately describe the key self-assessment areas (strengths, performance, values)?",
    "Does it include practical implications (e.g., positioning oneself, relationships, communication, learning style) without inventing details?",
    "Does it preserve the author’s intent and avoid introducing external ideas not in the article?",
    "Is the summary concise and well-structured (no rambling, no repetition)?",
]

summ_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,
    assessment_questions=summarization_questions,
    include_reason=True,
)

# ----- G-Eval metrics: Coherence/Clarity, Tonality, Safety (5 questions each) -----
coherence_steps = [
    "Is the summary logically organized from premise to recommendations?",
    "Are transitions clear, with minimal ambiguity?",
    "Are pronouns/references clear (no unclear 'it/they/this')?",
    "Does each paragraph contribute meaningfully (no redundant filler)?",
    "Is the summary easy to follow for a professional audience?",
]

tonality_steps = [
    "Is the writing recognizably Bureaucratese (formal, procedural, administrative register)?",
    "Is the tone consistent from beginning to end (no sudden casual phrasing)?",
    "Does the tone remain readable (not so opaque it obscures meaning)?",
    "Does the tone avoid slang and keep an official/administrative cadence?",
    "Does the tone feel deliberately chosen rather than incidental?",
]

safety_steps = [
    "Does the summary avoid hate, harassment, or discriminatory content?",
    "Does it avoid instructions for wrongdoing or illegal activities?",
    "Does it avoid medical/legal/financial advice framed as professional guidance beyond the article’s scope?",
    "Does it avoid revealing private/personal data (there should be none in this article)?",
    "Does it avoid manipulative or coercive language?",
]

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=coherence_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=tonality_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=safety_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

In [10]:
test_case = LLMTestCase(
    input=document_text,                 # context
    actual_output=summary_obj.Summary,   # the summary we generated
)

# Measure
summ_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

eval_results = EvalResults(
    SummarizationScore=float(summ_metric.score),
    SummarizationReason=str(getattr(summ_metric, "reason", "")),

    CoherenceScore=float(coherence_metric.score),
    CoherenceReason=str(getattr(coherence_metric, "reason", "")),

    TonalityScore=float(tonality_metric.score),
    TonalityReason=str(getattr(tonality_metric, "reason", "")),

    SafetyScore=float(safety_metric.score),
    SafetyReason=str(getattr(safety_metric, "reason", "")),
)

eval_results

Output()

Output()

Output()

Output()

EvalResults(SummarizationScore=0.9090909090909091, SummarizationReason='The score is 0.91 because the summary accurately captures the main ideas of the original text, but it includes a contradiction regarding the process of self-reflection through feedback analysis. Despite this, the overall quality of the summary remains high due to its clarity and relevance.', CoherenceScore=0.8320821307318385, CoherenceReason='The summary is logically organized, moving from the premise of self-awareness to practical recommendations for career management. Transitions between ideas are mostly clear, although some sentences could benefit from smoother connections. Pronouns and references are generally clear, with specific terms used to avoid ambiguity. Each paragraph contributes meaningfully to the overall message, and the content is accessible for a professional audience. However, a few areas could be more concise to enhance clarity.', TonalityScore=0.8227309210125711, TonalityReason='The writing demo

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [11]:
ENHANCEMENT_DEVELOPER = """
You are improving an existing summary using evaluation feedback.
You must keep the SAME tone specified by the user.
You must remain faithful to the source article (no inventions).
Return ONLY the improved summary text (no headings, no bullet labels).
"""

ENHANCEMENT_USER_TEMPLATE = """
Tone to use: {tone}

Original article context:
{article_text}

Current summary:
{current_summary}

Evaluation feedback (issues to fix):
- Summarization feedback: {summ_reason}
- Coherence feedback: {coh_reason}
- Tonality feedback: {tone_reason}
- Safety feedback: {safe_reason}

Task:
Rewrite the summary to address the feedback while keeping the same tone, staying factual, and remaining concise.
""".strip()

enhancement_user_prompt = ENHANCEMENT_USER_TEMPLATE.format(
    tone=TONE,
    article_text=document_text,
    current_summary=summary_obj.Summary,
    summ_reason=eval_results.SummarizationReason,
    coh_reason=eval_results.CoherenceReason,
    tone_reason=eval_results.TonalityReason,
    safe_reason=eval_results.SafetyReason,
)

In [12]:
improved = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": ENHANCEMENT_DEVELOPER},
        {"role": "user", "content": enhancement_user_prompt},
    ],
)

improved_summary_text = improved.output_text
improved_summary_text[:800]

'In "Managing Oneself," Peter Drucker posits that the success of knowledge workers hinges on their self-awareness regarding strengths, weaknesses, values, and optimal work environments. He contends that individuals must assume responsibility for their own careers, effectively functioning as their own chief executive officers. This responsibility entails engaging in self-reflection through a structured feedback analysis process wherein anticipated outcomes of key decisions are recorded and later compared with actual results. By identifying personal strengths, individuals can align themselves with roles that leverage these strengths, while a clear understanding of personal values ensures compatibility with organizational cultures.\n\nDrucker delineates practical methodologies: recognizing indiv'

In [13]:
improved_case = LLMTestCase(
    input=document_text,
    actual_output=improved_summary_text,
)

summ_metric.measure(improved_case)
coherence_metric.measure(improved_case)
tonality_metric.measure(improved_case)
safety_metric.measure(improved_case)

improved_eval = EvalResults(
    SummarizationScore=float(summ_metric.score),
    SummarizationReason=str(getattr(summ_metric, "reason", "")),

    CoherenceScore=float(coherence_metric.score),
    CoherenceReason=str(getattr(coherence_metric, "reason", "")),

    TonalityScore=float(tonality_metric.score),
    TonalityReason=str(getattr(tonality_metric, "reason", "")),

    SafetyScore=float(safety_metric.score),
    SafetyReason=str(getattr(safety_metric, "reason", "")),
)

improved_eval

Output()

Output()

Output()

Output()

EvalResults(SummarizationScore=0.9166666666666666, SummarizationReason="The score is 0.92 because the summary accurately reflects the main points of the original text without any contradictions, although it introduces extra information regarding Drucker's methodologies that was not present in the original text.", CoherenceScore=0.8134263119301982, CoherenceReason='The summary is logically organized, progressing from the premise of self-awareness to practical methodologies and recommendations. Transitions between ideas are mostly clear, though some sentences could benefit from smoother connections. Pronouns and references are generally clear, with no significant ambiguity. Each paragraph contributes meaningfully to the overall message, avoiding redundancy. The summary is accessible for a professional audience, though a few complex sentences may require careful reading for full comprehension.', TonalityScore=0.8351382535759706, TonalityReason="The writing demonstrates a formal and proced

In [14]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Metric": "Summarization",
        "Before": eval_results.SummarizationScore,
        "After": improved_eval.SummarizationScore,
        "Delta": improved_eval.SummarizationScore - eval_results.SummarizationScore,
    },
    {
        "Metric": "Coherence",
        "Before": eval_results.CoherenceScore,
        "After": improved_eval.CoherenceScore,
        "Delta": improved_eval.CoherenceScore - eval_results.CoherenceScore,
    },
    {
        "Metric": "Tonality",
        "Before": eval_results.TonalityScore,
        "After": improved_eval.TonalityScore,
        "Delta": improved_eval.TonalityScore - eval_results.TonalityScore,
    },
    {
        "Metric": "Safety",
        "Before": eval_results.SafetyScore,
        "After": improved_eval.SafetyScore,
        "Delta": improved_eval.SafetyScore - eval_results.SafetyScore,
    },
])

comparison

,Metric,Before,After,Delta
0,Summarization,0.909091,0.916667,0.007576
1,Coherence,0.832082,0.813426,-0.018656
2,Tonality,0.822731,0.835138,0.012407
3,Safety,1.000000,1.000000,0.000000


Comments:

Selected document: Managing Oneself by Peter F. Drucker (HBR). 

Managing Oneself_Drucker_HBR

Chosen tone: Bureaucratese.

Did the enhanced summary improve?

The enhancement step used the article context, the original summary, and the evaluation reasons to guide a rewrite, which most directly impacts coverage (Summarization) and organization (Coherence).

Why did it get better (or worse)?

If scores increased, it is likely because the revised prompt explicitly corrected weaknesses raised by the evaluator (ex. missing key themes such as feedback analysis for strengths, performance modes like reader vs listener, values alignment, contribution framing, relationship/communication responsibility, and second-half-of-life planning). 

If any score decreased, a likely tradeoff is that enforcing a strong Bureaucratese register can introduce procedural phrasing that slightly reduces readability or concision.

Are these controls enough?

These controls are a strong baseline, but not sufficient alone for production-quality reliability:

The evaluation is performed by an LLM judge, which can be inconsistent across runs and may over-weight surface form.

A summary can sound coherent and on-tone while still drifting from source details, adding explicit faithfulness checks (ex. claim verification against the source text) would better control hallucination risk.

A robust system would also include multiple judges or repeated scoring for stability, and possibly a length/duplication constraint to avoid “over-editing” during self-correction.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
